In [ ]:
pip install torch ultralytics numpy opencv-python scikit-image

# Adversarial Attack on YOLOv8n

A **disappearance attack**: PGD is used to suppress the confidence of every detection the clean image triggers, without any explicit constraint on what else the model might start seeing instead. That second half turns out to matter - see the Results section.

## 1. Setup: model and the attack

Uses the same two `scikit-image` test photos as the ResNet-18 notebook (`chelsea` - a cat, `coffee` - a cup on a table), so both image attacks are reproducible without external downloads.

YOLOv8's high-level `model.predict()` API runs under `torch.inference_mode()`, which blocks gradients entirely - the attack instead calls the underlying `model.model` (the raw `DetectionModel`) directly to get a differentiable forward pass.

In [ ]:
import json
import os

import cv2
import numpy as np
import torch
from skimage import data as skdata
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from ultralytics import YOLO
from ultralytics.data.augment import LetterBox

IMG_SIZE = 640
CONF_THRESH = 0.25
LETTERBOX = LetterBox((IMG_SIZE, IMG_SIZE), auto=False, scaleup=True)


def load_sample(name):
    """Match YOLO's own preprocessing: aspect-ratio-preserving resize + gray padding to a square."""
    arr = getattr(skdata, name)()  # HWC uint8
    return LETTERBOX(image=arr)


def to_tensor(img_hwc_uint8):
    return torch.from_numpy(img_hwc_uint8).permute(2, 0, 1).float().unsqueeze(0) / 255.0


def raw_forward(model, x):
    """Raw DetectionModel forward: returns preds shaped [1, 84, 8400] (4 box coords + 80 class
    scores, already sigmoided, per anchor point). The second tuple element is only used for training.
    """
    return model(x)[0]


def target_detections(preds, conf_thresh=CONF_THRESH):
    """Anchors in the clean pass whose top class score clears conf_thresh - what we'll suppress."""
    cls_scores = preds[0, 4:, :]  # [80, 8400]
    max_score, max_cls = cls_scores.max(dim=0)
    keep = (max_score > conf_thresh).nonzero(as_tuple=True)[0]
    return keep, max_cls[keep], max_score[keep]


def describe_detections(wrapper, img_hwc_uint8):
    """Human-readable detections via the normal (NMS-postprocessed) predict path."""
    results = wrapper.predict(img_hwc_uint8, verbose=False, conf=CONF_THRESH)
    r = results[0]
    dets = [(wrapper.model.names[int(b.cls.item())], float(b.conf.item())) for b in r.boxes]
    return sorted(dets, key=lambda d: -d[1])

In [ ]:
def run_pgd_attack(model, img_hwc_uint8, epsilon, alpha, num_iter):
    """Disappearance attack: suppress the confidence of every anchor the clean image triggers."""
    orig = to_tensor(img_hwc_uint8)
    with torch.no_grad():
        clean_preds = raw_forward(model, orig)
    target_idx, target_cls, _ = target_detections(clean_preds)

    adv = orig.clone().detach()
    for _ in range(num_iter):
        adv.requires_grad_(True)
        preds = raw_forward(model, adv)
        cls_scores = preds[0, 4:, :]
        loss = cls_scores[target_cls, target_idx].sum()
        grad = torch.autograd.grad(loss, adv)[0]

        with torch.no_grad():
            # Minimize confidence at the targeted anchors -> descend the gradient.
            adv = adv - alpha * grad.sign()
            perturbation = torch.clamp(adv - orig, min=-epsilon, max=epsilon)
            adv = torch.clamp(orig + perturbation, min=0, max=1)

    return adv.detach(), len(target_idx)

## 2. Load the detector

In [ ]:
wrapper = YOLO("yolov8n.pt")
model = wrapper.model
model.eval()

## 3. Run the attack across samples and perturbation budgets

Same two `epsilon` budgets as the ResNet-18 notebook: `mild` (8/255) and `strong` (16/255). Raw and adversarial images are saved under `samples/yolov8n/`.

In [ ]:
SAMPLES_DIR = "samples/yolov8n"
os.makedirs(SAMPLES_DIR, exist_ok=True)

SAMPLE_STEMS = ["chelsea", "coffee"]
CONFIGS = [
    {"name": "mild", "epsilon": 8 / 255, "alpha": 2 / 255, "num_iter": 10},
    {"name": "strong", "epsilon": 16 / 255, "alpha": 4 / 255, "num_iter": 10},
]

results = []
for stem in SAMPLE_STEMS:
    img = load_sample(stem)
    orig_dets = describe_detections(wrapper, img)
    cv2.imwrite(f"{SAMPLES_DIR}/{stem}_raw.png", cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

    for cfg in CONFIGS:
        print(f"running {stem} / {cfg['name']} ...")
        adv_tensor, n_targets = run_pgd_attack(model, img, cfg["epsilon"], cfg["alpha"], cfg["num_iter"])
        adv_np = adv_tensor.squeeze(0).permute(1, 2, 0).numpy()
        adv_uint8 = (adv_np * 255).astype(np.uint8)
        adv_dets = describe_detections(wrapper, adv_uint8)

        orig_np = img.astype(np.float32) / 255.0
        diff = adv_np - orig_np
        psnr = peak_signal_noise_ratio(orig_np, adv_np, data_range=1.0)
        ssim = structural_similarity(orig_np, adv_np, channel_axis=2, data_range=1.0)

        metrics = {
            "sample": stem,
            "config": cfg["name"],
            "epsilon_255": round(cfg["epsilon"] * 255),
            "targeted_anchors": n_targets,
            "original_detections": orig_dets,
            "adversarial_detections": adv_dets,
            "detections_before": len(orig_dets),
            "detections_after": len(adv_dets),
            "linf_perturbation_255": float(np.max(np.abs(diff)) * 255),
            "l2_perturbation": float(np.linalg.norm(diff)),
            "psnr_db": float(psnr),
            "ssim": float(ssim),
        }
        results.append(metrics)
        cv2.imwrite(f"{SAMPLES_DIR}/{stem}_{cfg['name']}_adversarial.png", cv2.cvtColor(adv_uint8, cv2.COLOR_RGB2BGR))

        print(f"[{stem} / {cfg['name']}] epsilon={metrics['epsilon_255']}/255  targeted {n_targets} anchors")
        print(f"  original detections:    {orig_dets}")
        print(f"  adversarial detections: {adv_dets}")
        print(f"  Linf={metrics['linf_perturbation_255']:.1f}/255  PSNR={psnr:.1f}dB  SSIM={ssim:.4f}")
        print()

with open(f"{SAMPLES_DIR}/results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
print("wrote results.json")

running chelsea / mild ...
[chelsea / mild] epsilon=8/255  targeted 7 anchors
  original detections:    [('cat', 0.6103441715240479)]
  adversarial detections: [('person', 0.9628097414970398), ('donut', 0.7061953544616699), ('person', 0.5096240043640137), ('chair', 0.4497954547405243), ('chair', 0.32702866196632385), ('wine glass', 0.2566416263580322), ('chair', 0.2557154595851898)]
  Linf=8.0/255  PSNR=34.2dB  SSIM=0.8118

running chelsea / strong ...
[chelsea / strong] epsilon=16/255  targeted 7 anchors
  original detections:    [('cat', 0.6103441715240479)]
  adversarial detections: [('cow', 0.8210317492485046)]
  Linf=16.0/255  PSNR=28.2dB  SSIM=0.5496

running coffee / mild ...
[coffee / mild] epsilon=8/255  targeted 23 anchors
  original detections:    [('cup', 0.9083733558654785), ('dining table', 0.46282678842544556)]
  adversarial detections: [('toilet', 0.7555314898490906), ('cat', 0.49297282099723816)]
  Linf=8.0/255  PSNR=34.2dB  SSIM=0.8086

running coffee / strong ...
[co

## 4. Results

Every configuration fully suppresses the true detection (`cat`, `cup`+`dining table`) - the disappearance objective works as intended in all 4 runs. But because nothing in the loss discourages the model from firing on some *other* class, every run also produces confident hallucinated detections (`person`, `donut`, `chair`, `cow`, `toilet`) that were never in the image - the strongest, `chelsea / mild`, hallucinates a `person` at 96% confidence. This is a fair reflection of how these attacks behave in practice: reliably breaking the original detection is easy, but controlling what the model reports *instead* needs an explicit constraint this attack doesn't have.

One more thing worth being upfront about: sign-based PGD (`torch.sign(grad)`) is numerically sensitive. Re-running this exact notebook logic against a slightly different PyTorch/torchvision build changed *which* hallucinated class showed up (though never whether the true detection vanished) - the specific class names above are honest outputs of the pinned versions in `requirements.txt`, not necessarily what you'd see with a different PyTorch build.

Raw and adversarial `.png` files are committed under `samples/yolov8n/` - open them on GitHub to compare directly.

In [ ]:
print(f"{'sample':<10}{'config':<8}{'eps/255':<9}{'before':<45}{'after':<45}")
for r in results:
    before = ", ".join(f"{c}:{p:.2f}" for c, p in r["original_detections"]) or "(none)"
    after = ", ".join(f"{c}:{p:.2f}" for c, p in r["adversarial_detections"]) or "(none)"
    print(f"{r['sample']:<10}{r['config']:<8}{r['epsilon_255']:<9}{before:<45}{after:<45}")

sample    config  eps/255  before                                       after                                        
chelsea   mild    8        cat:0.61                                     person:0.96, donut:0.71, person:0.51, chair:0.45, chair:0.33, wine glass:0.26, chair:0.26
chelsea   strong  16       cat:0.61                                     cow:0.82                                     
coffee    mild    8        cup:0.91, dining table:0.46                  toilet:0.76, cat:0.49                        
coffee    strong  16       cup:0.91, dining table:0.46                  cat:0.77                                     


## 5. Does explicitly suppressing hallucinations help?

The disappearance attack above only penalizes the anchors that fired in the *clean* pass - nothing stops the model from gaining confidence somewhere else as a side effect. A natural fix: also penalize the top-K most confident `(class, anchor)` pairs *anywhere in the image*, recomputed every iteration, not just the ones that were originally there.

In [ ]:
def run_pgd_attack_full_suppression(model, img_hwc_uint8, epsilon, alpha, num_iter,
                                     suppress_topk=10, suppress_weight=1.0):
    """Also suppress the top-K most confident (class, anchor) pairs anywhere, every iteration -
    not just the ones that fired in the clean pass - to stop the attack from just moving confidence
    to a different class or location instead of actually removing it."""
    orig = to_tensor(img_hwc_uint8)
    with torch.no_grad():
        clean_preds = raw_forward(model, orig)
    target_idx, target_cls, _ = target_detections(clean_preds)

    adv = orig.clone().detach()
    for _ in range(num_iter):
        adv.requires_grad_(True)
        preds = raw_forward(model, adv)
        cls_scores = preds[0, 4:, :]  # [80, 8400]

        target_loss = cls_scores[target_cls, target_idx].sum()
        global_max_scores, _ = cls_scores.max(dim=0)  # best class score per anchor, recomputed live
        topk_scores, _ = global_max_scores.topk(suppress_topk)
        suppression_loss = topk_scores.sum()

        loss = target_loss + suppress_weight * suppression_loss
        grad = torch.autograd.grad(loss, adv)[0]
        with torch.no_grad():
            adv = adv - alpha * grad.sign()
            perturbation = torch.clamp(adv - orig, min=-epsilon, max=epsilon)
            adv = torch.clamp(orig + perturbation, min=0, max=1)
    return adv.detach()


comparison = []
for stem in SAMPLE_STEMS:
    img = load_sample(stem)
    orig_dets = describe_detections(wrapper, img)

    for cfg in CONFIGS:
        adv_fix = run_pgd_attack_full_suppression(
            model, img, cfg["epsilon"], cfg["alpha"], cfg["num_iter"]
        )
        adv_fix_uint8 = (adv_fix.squeeze(0).permute(1, 2, 0).numpy() * 255).astype(np.uint8)
        dets_fix = describe_detections(wrapper, adv_fix_uint8)

        original_attack_dets = next(
            r["adversarial_detections"] for r in results if r["sample"] == stem and r["config"] == cfg["name"]
        )
        comparison.append({
            "sample": stem, "config": cfg["name"],
            "original_detections": orig_dets,
            "disappearance_only": original_attack_dets,
            "full_suppression": dets_fix,
        })
        print(f"[{stem} / {cfg['name']}]")
        print(f"  original:            {orig_dets}")
        print(f"  disappearance only:  {original_attack_dets}")
        print(f"  + full suppression:  {dets_fix}")
        print()

[chelsea / mild]
  original:            [('cat', 0.6103441715240479)]
  disappearance only:  [('person', 0.9628097414970398), ('donut', 0.7061953544616699), ('person', 0.5096240043640137), ('chair', 0.4497954547405243), ('chair', 0.32702866196632385), ('wine glass', 0.2566416263580322), ('chair', 0.2557154595851898)]
  + full suppression:  [('teddy bear', 0.3912785053253174), ('teddy bear', 0.3594324290752411), ('teddy bear', 0.29883337020874023), ('teddy bear', 0.27214714884757996)]

[chelsea / strong]
  original:            [('cat', 0.6103441715240479)]
  disappearance only:  [('cow', 0.8210317492485046)]
  + full suppression:  [('vase', 0.6172290444374084), ('vase', 0.5048828721046448), ('vase', 0.30075421929359436)]

[coffee / mild]
  original:            [('cup', 0.9083733558654785), ('dining table', 0.46282678842544556)]
  disappearance only:  [('toilet', 0.7555314898490906), ('cat', 0.49297282099723816)]
  + full suppression:  []

[coffee / strong]
  original:            [('cup'

**It helps, but it doesn't solve it.** Comparing max hallucinated confidence per run:

| sample | config | disappearance only | + full suppression |
|---|---|---|---|
| chelsea | mild | person, 96% | teddy bear, 39% |
| chelsea | strong | cow, 82% | vase, 62% |
| coffee | mild | toilet, 76% | *(nothing)* |
| coffee | strong | cat, 77% | cake, 45% |

The true detection still vanishes in all four runs, and adding the global suppression term roughly halves the average hallucinated confidence (and eliminates it outright for `coffee / mild`), for the same `epsilon` budget. But it hallucinates in 3 of 4 runs still - suppressing the top-K most confident anchors each iteration doesn't stop the model from finding some *11th*-most-confident region to push up instead. This is a real, honest result rather than a full fix: an untargeted PGD disappearance attack pushes probability mass somewhere, and no small set of extra loss terms fully pins down where. A complete fix would need to also suppress detections at *every* anchor above some threshold, not just the top-K, which starts to look less like an attack and more like an explicit multi-objective optimization problem.